# Ciencia de Datos — FAMAF 2026
## Predicción de `vote20cand` (voto presidencial 2020) — ANES Social Media 2020-2022

Pipeline reproducible y **listo para Colab**:
1. Carga de datos (desde el repo público, sin subir nada).
2. Limpieza de la label y de los códigos de faltantes.
3. Eliminación del *leakage* obvio (editable).
4. Baselines: Regresión Logística, LDA (Fisher), Random Forest, XGBoost.
5. Importancia de features: `feature_importances_`, *permutation importance* y **SHAP**.
6. Curva **precisión vs. cantidad de features** (selección por importancia).

> **GPU:** solo **XGBoost** la aprovecha (`device='cuda'`). RF / LogReg / LDA corren en CPU.
> En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU* y dejá `USE_GPU = True`.

### 1. Dependencias
En Colab `xgboost` ya viene; instalamos `shap` por las dudas.

In [ ]:
# Colab: instalar dependencias que puedan faltar
!pip -q install shap xgboost scikit-learn pandas matplotlib 2>/dev/null
print("OK")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
print("Imports OK")

### 2. Configuración
Acá se edita todo: la label, los códigos de faltantes, las columnas administrativas
y el **leakage obvio** (columnas que *son* el voto). Esta lista se ajusta con el codebook.

In [ ]:
# ---- Fuente de datos ----
RAW_URL = ("https://raw.githubusercontent.com/tomasgomezpizarro/proyecto-datos/"
           "master/anes_specialstudy_2020-2022_socialmedia_csv_20230705/"
           "anes_specialstudy_2020-2022_socialmedia_csv_20230705.csv")

TARGET   = "vote20cand"
USE_GPU  = True            # XGBoost en GPU (poné False si corrés en CPU)

# ANES codifica faltantes / 'no aplica' con valores negativos -> los pasamos a NaN
MISSING_CODES = [-1, -2, -3, -4, -5, -6, -7, -8, -9]

# Valores de la label que NO son un voto real (no aplica / sin respuesta) -> se descartan
TARGET_DROP_VALUES = [-1, -7, -8, -9]

# ---- Columnas administrativas / metadatos (no son features) ----
ADMIN_COLS = [
    "version", "caseid",
    "weight_pre", "weight_pre_nr", "weight_pre_spss",
    "weight_post", "weight_post_nr", "weight_post_spss",
    "w2qual", "start", "end", "duration", "surv_mode", "surv_lang",
    "device", "w1congdist", "w2congdist",
]

# ---- LEAKAGE OBVIO para vote20cand (EDITAR con el codebook) ----
# Columnas que prácticamente 'dictan' el voto 2020. Se detectan por nombre.
# Patrones: cualquier columna que empiece con estos prefijos se considera leakage.
LEAKAGE_PREFIXES = ["vote20", "ran_vote2cand", "voterep"]
# Columnas de leakage explícitas (agregar las que el codebook confirme):
LEAKAGE_EXPLICIT = ["primparty"]

print("Config lista. Target:", TARGET)

### 3. Carga de datos
El CSV está en **latin-1** (tiene bytes no-UTF8).

In [ ]:
df = pd.read_csv(RAW_URL, encoding="latin-1", low_memory=False)
print("Shape original:", df.shape)
df.head(3)

### 4. Limpieza de la label
Descartamos las filas donde `vote20cand` no es un voto real (`-1`, etc.).

In [ ]:
print("Distribucion ORIGINAL de", TARGET)
print(df[TARGET].value_counts(dropna=False).sort_index())

df = df[~df[TARGET].isin(TARGET_DROP_VALUES)].copy()
df = df.dropna(subset=[TARGET])
print("\nShape tras filtrar la label:", df.shape)
print("\nDistribucion FINAL de", TARGET)
print(df[TARGET].value_counts().sort_index())

# NOTA: mapear codigos -> nombres de candidato desde el codebook PDF para la presentacion.
# Ej (VERIFICAR en codebook): {1:'Biden', 5:'Trump', ...}
CANDIDATE_NAMES = {}  # completar

### 5. Construcción de `X`, `y`: leakage + faltantes
- Sacamos label, columnas administrativas y leakage.
- Convertimos los códigos negativos de faltante a `NaN`.
- Las columnas de texto residuales se codifican a números.

In [ ]:
# 1) detectar columnas de leakage por prefijo + explicitas
leak_cols = set(LEAKAGE_EXPLICIT)
for c in df.columns:
    if any(c.startswith(p) for p in LEAKAGE_PREFIXES):
        leak_cols.add(c)
leak_cols.discard(TARGET)
print(f"Columnas de leakage detectadas ({len(leak_cols)}):")
print(sorted(leak_cols))

# 2) columnas a descartar
drop_cols = set(ADMIN_COLS) | leak_cols | {TARGET}
feature_cols = [c for c in df.columns if c not in drop_cols]
print(f"\nFeatures candidatas: {len(feature_cols)}")

X = df[feature_cols].copy()
y_raw = df[TARGET].astype(int).copy()

# 3) faltantes: negativos -> NaN
X = X.replace(MISSING_CODES, np.nan)

# 4) columnas de texto -> codigos numericos
obj_cols = X.select_dtypes(include="object").columns
for c in obj_cols:
    X[c] = pd.factorize(X[c])[0]
    X[c] = X[c].replace(-1, np.nan)   # factorize pone -1 a los NaN

# 5) label codificada 0..k-1 (XGBoost lo necesita)
le = LabelEncoder()
y = le.fit_transform(y_raw)
print("\nClases (codigo original -> indice):", dict(zip(le.classes_, range(len(le.classes_)))))
print("X:", X.shape, "| y:", y.shape)

### 6. Split train/test
Estratificado para mantener la proporción de clases.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)

### 7. Baselines
LogReg / LDA / RF necesitan imputación (no toleran NaN). XGBoost maneja NaN nativo.

In [ ]:
results = {}

def evaluar(nombre, modelo, Xtr, Xte):
    modelo.fit(Xtr, y_train)
    pred = modelo.predict(Xte)
    acc = accuracy_score(y_test, pred)
    f1  = f1_score(y_test, pred, average="macro")
    results[nombre] = {"acc": acc, "f1_macro": f1, "modelo": modelo}
    print(f"{nombre:22s}  acc={acc:.3f}  f1_macro={f1:.3f}")
    return modelo

# --- imputacion para los modelos que no toleran NaN ---
imp = SimpleImputer(strategy="most_frequent")
X_train_imp = pd.DataFrame(imp.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_imp  = pd.DataFrame(imp.transform(X_test),  columns=X.columns, index=X_test.index)

# 1) Regresion Logistica (baseline)
logreg = Pipeline([("scaler", StandardScaler()),
                   ("clf", LogisticRegression(max_iter=2000, n_jobs=-1))])
evaluar("LogReg", logreg, X_train_imp, X_test_imp)

# 2) LDA (Fisher) - el 'tema que exploramos'
evaluar("LDA (Fisher)", LinearDiscriminantAnalysis(), X_train_imp, X_test_imp)

# 3) Random Forest
rf = RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=RANDOM_STATE)
evaluar("RandomForest", rf, X_train_imp, X_test_imp)

In [ ]:
# 4) XGBoost (GPU si esta disponible) - maneja NaN sin imputar
from xgboost import XGBClassifier

xgb_params = dict(
    n_estimators=600, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="multi:softprob", num_class=len(np.unique(y)),
    random_state=RANDOM_STATE, tree_method="hist",
)
if USE_GPU:
    xgb_params["device"] = "cuda"

try:
    xgb = XGBClassifier(**xgb_params)
    evaluar("XGBoost", xgb, X_train, X_test)   # con NaN, sin imputar
except Exception as e:
    print("GPU fallo, reintento en CPU:", e)
    xgb_params.pop("device", None)
    xgb = XGBClassifier(**xgb_params)
    evaluar("XGBoost", xgb, X_train, X_test)

In [ ]:
# Tabla comparativa + matriz de confusion del mejor
resumen = (pd.DataFrame({k: {"acc": v["acc"], "f1_macro": v["f1_macro"]}
                         for k, v in results.items()}).T
           .sort_values("f1_macro", ascending=False))
print(resumen)

best_name = resumen.index[0]
best = results[best_name]["modelo"]
Xte_best = X_test if best_name == "XGBoost" else X_test_imp
ConfusionMatrixDisplay.from_predictions(y_test, best.predict(Xte_best))
plt.title(f"Matriz de confusion - {best_name}")
plt.show()

### 8. Importancia de features (3 niveles)
**Recordatorio:** alta importancia = *alarma* de posible leakage, no sentencia.
Si una feature aparece arriba de todo, revisarla en el codebook.

In [ ]:
# Nivel 1: feature_importances_ (impureza) de RF y XGB
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (nombre, modelo, cols) in zip(axes, [
        ("RandomForest", rf, X.columns), ("XGBoost", xgb, X.columns)]):
    imp_s = pd.Series(modelo.feature_importances_, index=cols).sort_values(ascending=False).head(20)
    imp_s[::-1].plot.barh(ax=ax)
    ax.set_title(f"Top 20 - {nombre} (feature_importances_)")
plt.tight_layout(); plt.show()

In [ ]:
# Nivel 2: Permutation importance (mas honesta) sobre el TEST, usando RF
perm = permutation_importance(rf, X_test_imp, y_test, n_repeats=10,
                              random_state=RANDOM_STATE, scoring="f1_macro", n_jobs=-1)
perm_s = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(20)
perm_s[::-1].plot.barh(figsize=(8, 7))
plt.title("Top 20 - Permutation importance (RF, test, f1_macro)")
plt.tight_layout(); plt.show()

In [ ]:
# Nivel 3: SHAP (descompone cada prediccion). TreeExplainer es rapido sobre XGB.
import shap
expl = shap.TreeExplainer(xgb)
X_shap = X_test.sample(min(500, len(X_test)), random_state=RANDOM_STATE)
shap_values = expl.shap_values(X_shap)
shap.summary_plot(shap_values, X_shap, max_display=20, show=True)

### 9. Curva precisión vs. cantidad de features
La idea que discutimos: rankear por importancia y entrenar con las top-k, viendo cómo
cae (o no) la métrica. Usa **XGBoost (GPU)**, así que es rápido.
> Versión rigurosa alternativa: `sklearn.feature_selection.RFECV` (más lenta, CPU).

In [ ]:
# Ranking global por importancia de XGB
rank = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
ks = [5, 10, 20, 30, 50, 75, 100, 150, 200, 300, min(400, X.shape[1]), X.shape[1]]
ks = sorted(set(k for k in ks if k <= X.shape[1]))

curva = []
for k in ks:
    cols_k = rank.index[:k].tolist()
    m = XGBClassifier(**xgb_params)
    m.fit(X_train[cols_k], y_train)
    pred = m.predict(X_test[cols_k])
    curva.append({"k": k,
                  "acc": accuracy_score(y_test, pred),
                  "f1_macro": f1_score(y_test, pred, average="macro")})
curva = pd.DataFrame(curva)
print(curva)

plt.figure(figsize=(9, 5))
plt.plot(curva["k"], curva["acc"], "o-", label="accuracy")
plt.plot(curva["k"], curva["f1_macro"], "s-", label="f1_macro")
plt.xlabel("Cantidad de features (top-k por importancia)")
plt.ylabel("Metrica en test"); plt.title("Precision vs. cantidad de features")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 10. Notas para la presentación
- **Leakage:** se eliminó el obvio por nombre (`vote20*`, `ran_vote2cand*`, `voterep`, `primparty`).
  Revisar el **codebook** las features que queden con importancia sospechosamente alta.
- **Encoding:** por ahora los códigos se usan como números (suficiente para árboles; LogReg/LDA lo
  toleran como baseline). Mejora futura: one-hot / target-encoding de las categóricas reales.
- **Desbalance:** `vote20cand` está desbalanceada → mirar `f1_macro` y la matriz de confusión,
  no solo accuracy. Probar `class_weight` / SMOTE (vistos en clase 12).
- **Mapear códigos → nombres de candidato** desde el codebook para los gráficos finales.